# Notebook 05: Discusión y conclusiones

---

### Entorno y persistencia de resultados

La celda siguiente fija tres cosas que condicionan la reproducibilidad del experimento:

**Semilla fija.** `RANDOM_STATE = 42` se aplica a la partición train/test y al ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Los resultados se escriben en `MyDrive/hotel_booking` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. Esto permite que el experimento se ejecute
en varias sesiones sin repetir etapas: el notebook 03 deja las particiones preparadas y los notebooks
04 y 05 las consumen tal cual, garantizando que todos operan exactamente sobre los mismos datos.
Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir en una ubicación
volátil.

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. Se invoca siempre antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib
y el archivo resultante quedaría en blanco.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/hotel_booking"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath("./hotel_booking")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Grafico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando local")
print("Carpeta de trabajo:", RUTA)
print("Graficos (splits) :", CARPETA_SPLITS)

In [ ]:
import joblib
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

paquete = joblib.load(os.path.join(RUTA, "datos_preparados.joblib"))
salida = joblib.load(os.path.join(RUTA, "modelo_y_resultados.joblib"))

X_train, X_test = paquete["X_train"], paquete["X_test"]
y_train, y_test = paquete["y_train"], paquete["y_test"]
modelo_final = salida["modelo_final"]
resultados = salida["resultados"]
tabla_peso = salida["tabla_peso"]
pred_train, pred_test = salida["pred_train"], salida["pred_test"]
etiquetas = salida["etiquetas"]

print("Mejor configuracion:", salida["mejor"]["estrategia"], "| C =", salida["mejor"]["C"])

## 4.1 Interpretación y comparación de métricas train vs. test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, estrategia in zip(axes, ["Softmax", "One-vs-Rest"]):
    sub = resultados[resultados["estrategia"] == estrategia].sort_values("C")
    ax.plot(sub["C"], sub["f1_macro_train"], "o-", label="Train")
    ax.plot(sub["C"], sub["f1_macro_test"], "s-", label="Test")
    ax.set_xscale("log")
    ax.set_xlabel("C  (menor = mas regularizacion)")
    ax.set_ylabel("F1 macro")
    ax.set_title(f"{estrategia}: regularizacion vs desempeno")
    ax.legend()

plt.tight_layout()
guardar("05_regularizacion_vs_desempeno")
plt.show()

print(resultados.sort_values(["estrategia", "C"])[
    ["estrategia", "C", "f1_macro_train", "f1_macro_test", "brecha"]
].round(4).to_string(index=False))

In [ ]:
print("=== Accuracy frente a F1 macro en el modelo elegido ===")
print(f"Accuracy test : {accuracy_score(y_test, pred_test):.4f}")
print(f"F1 macro test : {f1_score(y_test, pred_test, average='macro'):.4f}")
print("\n=== Efecto de la ponderacion por clase ===")
print(tabla_peso.round(4).to_string(index=False))

### Cada métrica en su contexto

El *recall* de `No-Show`, de 0,481, indica que el modelo anticipa cerca de la mitad de las reservas
que efectivamente terminaron en no presentación. Es la métrica que el hotel emplearía para decidir a
quién exigir garantía. La *precisión* de esa misma clase, de 0,037, indica que de cada cien reservas
señaladas como riesgosas menos de cuatro lo eran realmente, lo que se traduce en clientes confiables
a los que se impondría una condición innecesaria. Existe un intercambio directo entre ambas métricas:
elevar el recall implica emitir más alertas, y cada alerta adicional infundada tiene un costo
comercial.

### La distancia entre accuracy y F1 macro

La accuracy de prueba es 0,7282 y el F1 macro es 0,5354, una diferencia cercana a veinte puntos. La
brecha no constituye un defecto del cálculo sino la consecuencia esperada de un conjunto
desbalanceado. La accuracy pondera cada observación por igual, de modo que queda determinada por las
dos clases que concentran el 99 % de los casos; el F1 macro promedia las tres clases con el mismo
peso, de modo que el desempeño pobre sobre `No-Show` arrastra el resultado global.

Esta diferencia constituye la verificación empírica del criterio adoptado en la formulación del
problema, donde se estableció el F1 macro como métrica contractual antes de observar los datos. La
elección no se justificó retrospectivamente: se anticipó que un modelo capaz de ignorar la clase
minoritaria obtendría una accuracy alta y resultaría inservible, y los resultados lo confirman.

### Diagnóstico: no hay sobreajuste, hay saturación

La brecha entre entrenamiento y prueba en F1 macro es de 0,0069, y se mantiene en el mismo orden de
magnitud para las cinco configuraciones de `C` ensayadas en ambas estrategias. Un modelo
sobreajustado mostraría una curva de entrenamiento que se despega progresivamente de la de prueba a
medida que aumenta la flexibilidad; en cambio, ambas curvas permanecen paralelas y separadas por
menos de un punto porcentual.

El barrido de `C` resulta además esencialmente plano. Entre C = 0,01 y C = 10 el F1 macro de prueba
del modelo One-vs-Rest varía tres diezmilésimas. Reducir la regularización, es decir aumentar la
capacidad efectiva del modelo, no aporta mejora alguna.

El diagnóstico correspondiente es **underfitting por limitación de la representación** y no
sobreajuste. El techo de desempeño no lo impone la complejidad del modelo sino la información
contenida en las variables disponibles. La única configuración claramente inferior es C = 0,001, con
F1 macro de 0,5310, donde la regularización es tan fuerte que suprime señal útil.

### Softmax frente a One-vs-Rest

La estrategia One-vs-Rest supera a Softmax en las cinco configuraciones ensayadas, con una diferencia
sostenida de aproximadamente 0,022 de F1 macro: el mejor Softmax alcanza 0,5133 frente a 0,5354 del
mejor One-vs-Rest. La ventaja es modesta pero sistemática, lo que descarta que se trate de
variabilidad aleatoria.

Una explicación plausible es que, al entrenar un clasificador binario independiente por clase,
One-vs-Rest permite que la frontera de `No-Show` se ajuste sin competir directamente contra las dos
clases mayoritarias dentro de una misma función softmax normalizada, condición favorable en un
escenario de desbalance extremo.

### Mitigación de las limitaciones observadas

Dado que el problema es de underfitting y no de sobreajuste, reducir `C` no aportaría mejoras. Las
vías razonables consisten en ampliar la representación, incorporando variables nuevas o construyendo
términos de interacción, antes que en modificar la regularización. Si al agregar esas variables se
detectara sobreajuste, correspondería reducir `C` o adoptar penalización L1 con el solver `saga`, que
además realiza selección de variables llevando coeficientes a cero.

## 4.2 Análisis de la matriz de confusión y errores por clase

In [ ]:
cm_norm = confusion_matrix(y_test, pred_test, labels=etiquetas, normalize="true")

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="RdYlGn",
            xticklabels=etiquetas, yticklabels=etiquetas, ax=ax, vmin=0, vmax=1)
ax.set_title("Matriz de confusion normalizada por fila (TEST)")
ax.set_xlabel("Prediccion"); ax.set_ylabel("Valor real")
plt.tight_layout()
guardar("05_matriz_confusion_normalizada")
plt.show()

print("Cada fila suma 100 %: muestra en que se convirtio cada clase real.\n")
print(pd.DataFrame(
    confusion_matrix(y_test, pred_test, labels=etiquetas),
    index=[f"real {e}" for e in etiquetas],
    columns=[f"pred {e}" for e in etiquetas],
))

### El costo de las alertas de No-Show

La diagonal cuenta los aciertos, pero el hotel convive con la **columna** completa: todas las
reservas a las que el modelo le pondría una condición extra.

In [ ]:
cm = confusion_matrix(y_test, pred_test, labels=etiquetas)
i_ns = etiquetas.index("No-Show")
columna = cm[:, i_ns]
total_alertas = columna.sum()

print(f"Reservas marcadas como riesgo de No-Show: {total_alertas}")
for j, e in enumerate(etiquetas):
    print(f"   de las cuales eran realmente {e:<10}: {columna[j]:>5}  ({columna[j]/total_alertas:.1%})")

print(f"\nNo-Shows reales en el test: {cm[i_ns].sum()}")
print(f"Detectados: {cm[i_ns, i_ns]} ({cm[i_ns, i_ns]/cm[i_ns].sum():.1%})")
print(f"\nPor cada no-show detectado, el hotel molesta a "
      f"{(total_alertas - cm[i_ns, i_ns]) / cm[i_ns, i_ns]:.0f} clientes que si habrian venido "
      f"o que habrian avisado.")

Por cada no-show efectivamente detectado, el hotel exigiría una garantía a veintiséis clientes que
habrían llegado o que habrían avisado con anticipación. Esa proporción traduce una precisión de 0,037
en una consecuencia operativa concreta.

Determinar si el resultado es aceptable excede el alcance del modelo. Depende de la relación entre el
costo de una noche perdida sin posibilidad de reventa y el costo comercial de imponer una garantía a
un cliente confiable, magnitudes que corresponden a la administración del hotel. El aporte del
análisis consiste en cuantificar ese intercambio, no en resolverlo.

### Distribución de los errores

Las dos clases mayoritarias se clasifican razonablemente bien y se confunden entre sí de forma casi
simétrica: el 18,03 % de las cancelaciones se predice como `Check-Out` y el 11,69 % de las estadías
concretadas se predice como `Canceled`. Esa confusión mutua resulta esperable, dado que ambas clases
comparten la mayor parte del espacio de variables y se separan únicamente por diferencias de grado en
`lead_time`, pedidos especiales e historial.

### El error dominante en No-Show

De las 241 reservas que terminaron en no presentación, el modelo identifica correctamente 116, es
decir el 48,13 %. De las restantes, 80 se clasifican como `Check-Out` (33,20 %) y solo 45 como
`Canceled` (18,67 %).

Este resultado contradice la hipótesis intuitiva de que una no presentación se confundiría
principalmente con una cancelación, por tratarse en ambos casos de una reserva que no se concreta. El
error dominante va en la dirección opuesta: el modelo tiende a confundir el no-show con una estadía
normal.

El análisis descriptivo del notebook 02 explica el resultado. Una reserva destinada a terminar en no
presentación presenta la anticipación más corta de las tres clases, con mediana de 30 días frente a
116 en `Canceled`, y un historial de cancelaciones tan limpio como el de quien sí se hospeda, con
0,01 frente a 0,21. Al momento de tomarse no se asemeja a una cancelación sino a una reserva
ordinaria.

La razón de fondo es de naturaleza temporal. Cancelar es una decisión que el huésped toma y comunica,
asociada a rasgos observables desde el inicio: mucha anticipación, canal de agencia, antecedentes de
cancelación. No presentarse, en cambio, es una decisión tomada con posterioridad a la reserva y nunca
comunicada, de modo que no deja rastro en los datos disponibles al momento de reservar. El modelo no
puede anticipar un evento cuya causa todavía no ha ocurrido.

### La única señal disponible para la clase minoritaria

El coeficiente más alto hacia `No-Show` corresponde a `country = PRT`, con valor de 2,136. Portugal
es el mercado local de ambos hoteles, y la interpretación es económica: un huésped que reside cerca
pierde mucho menos por no presentarse que uno que ya incurrió en el costo de un vuelo internacional,
para quien el desplazamiento representa una inversión hundida.

Se trata, sin embargo, de una señal demográfica y no conductual. Identifica una población con mayor
propensión al no-show, pero no distingue, dentro de esa población, cuál reserva concreta terminará
así. Esto explica que la clase se detecte con recall moderado y precisión muy baja.

### Vínculo con el desbalance y con la separabilidad

Con 1.206 casos sobre 119.209, la clase `No-Show` aporta muy pocos ejemplos para estimar su frontera
de decisión. La ponderación `class_weight="balanced"` corrige el peso relativo del error durante el
entrenamiento, pero no genera información que no exista en los datos.

A lo anterior se suma la limitación específica del modelo lineal: la regresión logística traza
fronteras que son hiperplanos y solo captura relaciones entre variables si estas se construyen
explícitamente como términos nuevos. Si la distinción entre no presentarse y cancelar dependiera de
una combinación de condiciones, por ejemplo mercado local junto con estadía corta y ausencia de
pedidos especiales, el modelo actual no dispone de forma de representarla.
- **Limitación del modelo lineal.** La regresión logística traza fronteras que son hiperplanos. Si la
  distinción dependiera de interacciones (por ejemplo, mucha anticipación **y además** sin depósito
  **y además** cliente no recurrente) un modelo lineal solo puede capturarlas si esas interacciones
  se construyen explícitamente como variables nuevas.

### Coeficientes: qué aprendió el modelo

La ventaja del modelo lineal es que cada variable tiene un peso legible. Un coeficiente positivo para
una clase significa que esa variable empuja la predicción hacia ella.

In [ ]:
try:
    nombres = modelo_final.named_steps["prep"].get_feature_names_out()
    clf = modelo_final.named_steps["clf"]
    coefs = clf.coef_ if hasattr(clf, "coef_") else np.vstack([e.coef_ for e in clf.estimators_])
    clases = clf.classes_ if hasattr(clf, "classes_") else modelo_final.classes_

    tabla_coef = pd.DataFrame(coefs.T, index=nombres, columns=clases)
    for clase in clases:
        print(f"\n=== Variables que mas empujan hacia {clase} ===")
        print(tabla_coef[clase].sort_values(ascending=False).head(8).round(3))

    print("\n=== Donde queda lead_time ===")
    print(tabla_coef.loc[[i for i in tabla_coef.index if "lead_time" in i]].round(3))
except Exception as e:
    print("No se pudieron extraer los coeficientes:", e)

Los coeficientes obtenidos admiten cuatro lecturas de dominio.

**`deposit_type = Non Refund` es el predictor más fuerte del modelo**, con +4,043 hacia `Canceled`.
Su dirección es contraintuitiva, dado que la condición diseñada para desalentar la cancelación
aparece como su indicador principal. La lectura razonable no es causal sino de selección: el hotel ya
exige esa condición a las reservas que considera riesgosas, de modo que el modelo aprendió a
reconocer la política comercial del hotel antes que un efecto de esa política sobre el comportamiento
del huésped. La detección de esta confusión fue posible únicamente porque el modelo es interpretable;
un clasificador opaco habría incorporado la misma relación sin ofrecer forma de advertirla.

**`required_car_parking_spaces` es el predictor más fuerte de `Check-Out`**, con +1,811. Quien
reserva estacionamiento ha resuelto previamente cómo llegar al hotel, señal de compromiso disponible
desde el momento de la reserva.

**`lead_time` presenta coeficiente positivo hacia `Canceled` (+0,576) y negativo hacia `Check-Out`
(−0,570)**, en la dirección que anticipaba el análisis descriptivo, aunque con magnitud menor que las
dos variables anteriores.

**`country = PRT` es el predictor más fuerte de `No-Show`**, con +2,136. Portugal es el mercado local
de ambos hoteles: un huésped que reside cerca pierde menos por no presentarse, mientras que quien
viajó desde otro país ya asumió el costo del desplazamiento. Constituye el único indicio que el
modelo encuentra para esa clase, lo que explica su capacidad limitada de detectarla, dado que la
procedencia por sí sola no identifica cuál reserva concreta terminará en no presentación.

## 5.1 Conclusiones

Cada conclusión responde a un objetivo específico declarado en el notebook 01.

**Primera.** Se construyó y evaluó un clasificador multiclase basado en regresión logística que
predice el estado final de una reserva hotelera empleando exclusivamente información disponible en el
momento de reservar. La configuración seleccionada, One-vs-Rest con C = 0,1, alcanza un F1 macro de
0,5354 y una accuracy de 0,7282 sobre un conjunto de prueba independiente de 23.842 reservas. El
contraste con la línea base es el siguiente:

| Referencia | Accuracy | F1 macro |
|---|---|---|
| Predecir siempre la clase mayoritaria | 0,629 | 0,257 |
| Modelo obtenido | **0,728** | **0,535** |

La comparación arroja lecturas distintas según la métrica considerada. En accuracy la mejora es de
diez puntos, margen que podría parecer modesto. En F1 macro el modelo más que duplica la línea base.
Esa divergencia sostiene la elección metodológica adoptada en la formulación: la métrica definida al
plantear el problema es la que refleja el valor real del modelo, y no fue seleccionada después de
conocer los resultados.

**Segunda.** Las variables más informativas resultan accionables, en el sentido de que el hotel
dispone de todas ellas al recibir la reserva. Los coeficientes de mayor magnitud son
`deposit_type = Non Refund` (+4,043 hacia `Canceled`), `previous_cancellations` (+2,171),
`country = PRT` (+2,047 hacia `Canceled` y +2,136 hacia `No-Show`),
`required_car_parking_spaces` (+1,811 hacia `Check-Out`) y `lead_time` (+0,576 y −0,570). El caso del
depósito no reembolsable, analizado en la sección anterior, ilustra el valor de un modelo
interpretable: permitió identificar que la relación aprendida refleja la política comercial del hotel
y no un efecto causal sobre el comportamiento del huésped.

**Tercera.** La clase `No-Show` no se predice de manera confiable, con recall de 0,481 y precisión de
0,037. La causa no se reduce a su representación del 1 %. El análisis descriptivo mostró que una
reserva destinada a terminar en no presentación es indistinguible de una reserva ordinaria en el
momento de tomarse, porque la decisión de no presentarse se produce con posterioridad y nunca se
comunica. Constituye una limitación del problema tal como está planteado y no una deficiencia del
modelo lineal: ningún clasificador puede anticipar un evento cuya causa aún no ha ocurrido y del que
no existe registro en las variables disponibles.

**Cuarta.** Pese a esa limitación, el modelo cumple el objetivo de negocio planteado en la
formulación. Identifica el perfil de cancelación con un F1 de 0,731 y con anticipación suficiente
para que el hotel actúe, ya sea exigiendo depósito, aplicando una política de cancelación con plazo o
reservando capacidad para reventa. Dado que la cancelación con aviso permite recuperar la habitación,
resolver adecuadamente esta clase, que representa el 36 % de las reservas, tiene un impacto operativo
mayor que resolver una clase que concentra el 1 %.

### Limitaciones del enfoque lineal

La regresión logística asume fronteras de decisión que son hiperplanos y no captura interacciones
entre variables salvo que estas se construyan explícitamente. Dos observaciones del experimento
indican, no obstante, que la restricción del modelo no constituye el factor limitante principal.

La primera es que el barrido de `C` resultó esencialmente plano: entre C = 0,01 y C = 10 el F1 macro
de prueba varía tres diezmilésimas, de modo que aumentar la capacidad efectiva del modelo no mejora
el resultado. La segunda es que la brecha entre entrenamiento y prueba se mantiene en 0,0069, muy por
debajo de lo que indicaría sobreajuste. El modelo no memoriza: está saturado por la información
disponible.

A ello se suma que One-vs-Rest supera a Softmax de forma consistente pero por un margen reducido,
aproximadamente 0,022 de F1 macro. En conjunto, los tres indicios apuntan a que el techo lo impone la
representación de los datos y no la familia de modelos elegida. Sustituir la regresión logística por
un clasificador más flexible podría producir mejoras marginales, pero no resolvería la ausencia de
variables que separen `No-Show` de las demás clases.

### Recomendaciones de mejora

**Incorporar variables del período previo a la llegada.** Confirmaciones respondidas, contactos con
el hotel, modificaciones tardías o el momento en que se completó el pago aportarían señal sobre el
intervalo en el que se toma la decisión de no presentarse. Es la vía con mayor potencial, porque
ataca la causa identificada en la tercera conclusión.

**Construir términos de interacción.** Productos como `lead_time × deposit_type` o
`country × customer_type` permitirían al modelo lineal representar relaciones combinadas sin
renunciar a la interpretabilidad de sus coeficientes.

**Ajustar el umbral de decisión según el costo del error.** El criterio de máxima probabilidad supone
implícitamente que los tres tipos de error cuestan lo mismo, supuesto que no se sostiene en este
dominio. Calibrar el umbral de `No-Show` a partir de la relación entre el costo de una noche perdida
y el de una garantía innecesaria permitiría situar el modelo en el punto de operación que convenga al
hotel, sin necesidad de reentrenarlo.

**Validar la estabilidad temporal.** El conjunto abarca reservas entre julio de 2015 y agosto de
2017. Verificar que los coeficientes se mantengan al entrenar sobre un período y evaluar sobre el
siguiente permitiría estimar con qué frecuencia sería necesario reajustar el modelo en operación.

---

## Gráficos generados

Los siete gráficos de la serie quedan guardados en PNG a 200 dpi dentro de
`MyDrive/hotel_booking/splits/`, listos para insertar en el informe escrito.
La celda siguiente verifica que estén todos.

In [ ]:
esperados = [
    "02_boxplots_por_clase",
    "02_leadtime_y_tipo_deposito",
    "03_distribucion_de_clases",
    "04_matriz_confusion_train",
    "04_matriz_confusion_test",
    "05_regularizacion_vs_desempeno",
    "05_matriz_confusion_normalizada",
]

print("Carpeta:", CARPETA_SPLITS, "\n")
faltan = []
for nombre in esperados:
    ruta = os.path.join(CARPETA_SPLITS, nombre + ".png")
    if os.path.exists(ruta):
        print(f"  OK     {nombre}.png  ({os.path.getsize(ruta)/1024:.0f} KB)")
    else:
        print(f"  FALTA  {nombre}.png  -> ejecuta el notebook {nombre[:2]}")
        faltan.append(nombre)

print()
print(f"{len(esperados) - len(faltan)} de {len(esperados)} graficos disponibles.")